# 18 · Vector quantization: VQ-VAE codebooks, the straight-through estimator, FSQ

**Needed for:** VQ-VAE (vision track, step V7) and Janus (step V8). Janus generates images as a sequence of
discrete codes from a VQ tokenizer, predicted by the same language model that reads text.

To let a language model *generate* images token by token, images must become discrete tokens. A VQ tokenizer:
1. Encodes an image to a grid of vectors $z_e$.
2. Replaces each vector with its **nearest codebook entry** $e_k$; $k$ is the token id.
3. Decodes the quantized grid back to pixels.

Nearest-neighbour lookup has no gradient, so training needs two tricks: the **straight-through estimator** for the
encoder, and extra loss terms that pull codebook entries and encoder outputs toward each other.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. The VQ layer

$$\mathcal{L}_{VQ} = \underbrace{\lVert \text{sg}[z_e] - e_k\rVert^2}_{\text{codebook: move codes to encoder outputs}} + \beta\underbrace{\lVert z_e - \text{sg}[e_k]\rVert^2}_{\text{commitment: keep encoder near its code}}$$

$\text{sg}$ = stop-gradient (`.detach()`), $\beta = 0.25$ in the VQ-VAE paper.

**Straight-through:** `z_q = z_e + (e_k - z_e).detach()`. The forward value is $e_k$; the backward pass treats
quantization as the identity, so the decoder's gradient flows straight into the encoder.

In [ ]:
def vq_forward(z_e, codebook, beta=0.25):
    """z_e: (N, d), codebook: (K, d). Returns (z_q, indices, vq_loss)."""
    dist = (z_e.pow(2).sum(1, keepdim=True) - 2 * z_e @ codebook.T + codebook.pow(2).sum(1)[None])   # (N, K)
    idx = dist.argmin(1)
    e = codebook[idx]
    loss = F.mse_loss(e, z_e.detach()) + beta * F.mse_loss(z_e, e.detach())
    z_q = z_e + (e - z_e).detach()
    return z_q, idx, loss


def test_vq(fn):
    torch.manual_seed(0)
    codebook = torch.randn(16, 4, requires_grad=True)
    z_e = torch.randn(10, 4, requires_grad=True)
    z_q, idx, loss = fn(z_e, codebook)
    assert torch.equal(idx, torch.cdist(z_e, codebook).argmin(1))
    torch.testing.assert_close(z_q, codebook[idx])                        # forward value = the code
    (z_q * torch.arange(4.0)).sum().backward(retain_graph=True)
    torch.testing.assert_close(z_e.grad, torch.arange(4.0).expand(10, 4))  # straight-through: identity gradient
    z_e.grad = None
    codebook.grad = None
    loss.backward()
    assert codebook.grad is not None and codebook.grad.abs().sum() > 0     # the codebook term trains the codes
    torch.testing.assert_close(z_e.grad, 0.25 * 2 * (z_e - codebook[idx]).detach() / z_e.numel())  # commitment only


test_vq(vq_forward)
print("VQ layer: nearest code forward, identity gradient backward, codebook + commitment losses")

## 2. Codebook usage: perplexity and dead codes

A common failure: most codes are never chosen ("dead codes"), wasting vocabulary. Track **perplexity** of code
usage, $\exp(H(p))$, which equals $K$ when all $K$ codes are used equally and 1 when only one is used.
Fixes: initialize codes from encoder outputs (k-means), re-seed dead codes with random encoder outputs, update
codes with an exponential moving average instead of gradients, or use lower-dimensional / L2-normalized codes.

In [ ]:
def perplexity(idx, K):
    p = torch.bincount(idx, minlength=K).float() / idx.numel()
    return torch.exp(-(p * (p + 1e-10).log()).sum()).item()


print("all 8 codes equally:", round(perplexity(torch.arange(8).repeat(100), 8), 2),
      "| only 2 codes:", round(perplexity(torch.tensor([0, 1] * 400), 8), 2))

## 3. Toy VQ autoencoder

Data: a ring of 8 tight Gaussian clusters in 2-D (as in notebooks 16–17), and 16 codes. Without help, codes that
lose early are never chosen again. Typically fewer codes survive than there are clusters, so some clusters share a
code and their reconstructions land between them. The dead-code reset reassigns any code unused in a batch to a
random encoder output. Each setting runs with 5 seeds, since single runs vary a lot.

In [ ]:
CENTERS = torch.stack([torch.tensor([math.cos(2 * math.pi * i / 8), math.sin(2 * math.pi * i / 8)]) * 2 for i in range(8)])


def sample(n):
    return CENTERS[torch.randint(0, 8, (n,))] + 0.05 * torch.randn(n, 2)


def train_vq(reset_dead, steps=2000, K=16, seed=0):
    torch.manual_seed(seed)
    enc = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 2))
    dec = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 2))
    codebook = nn.Parameter(torch.randn(K, 2) * 0.1)
    opt = torch.optim.Adam([*enc.parameters(), *dec.parameters(), codebook], lr=3e-3)
    for step in range(steps):
        x = sample(256)
        z_e = enc(x)
        z_q, idx, vq_loss = vq_forward(z_e, codebook)
        loss = F.mse_loss(dec(z_q), x) + vq_loss
        opt.zero_grad()
        loss.backward()
        opt.step()
        if reset_dead and step % 100 == 0:
            with torch.no_grad():
                dead = torch.bincount(idx, minlength=K) == 0
                codebook[dead] = z_e[torch.randint(0, len(z_e), (int(dead.sum()),))].detach()
    with torch.no_grad():
        x = sample(4000)
        _, idx, _ = vq_forward(enc(x), codebook)
        recon = F.mse_loss(dec(codebook[idx]), x).item()
    return recon, perplexity(idx, K), (torch.bincount(idx, minlength=K) > 0).sum().item()


for reset in (False, True):
    runs = [train_vq(reset, steps=1000, seed=s) for s in range(5)]
    print(f"dead-code reset={str(reset):5s}  (5 seeds)")
    print("   reconstruction MSE:", [round(r[0], 4) for r in runs])
    print("   codes used of 16:  ", [r[2] for r in runs])

## 4. Finite scalar quantization (FSQ): no codebook at all

FSQ (Mentzer et al., 2023) projects to a few dimensions (say 4), bounds each with `tanh`, and rounds each dimension
to one of $L$ levels. The implicit codebook is the grid: $L^4$ codes, e.g. $8^4 = 4096$. No codebook loss,
no commitment loss, no dead codes; straight-through rounding is the only trick.

In [ ]:
def fsq(z, levels=8):
    """z: (N, d). Each dim rounded to `levels` evenly spaced values in [-1, 1]. Returns (z_q, code ids)."""
    u = (torch.tanh(z) + 1) / 2 * (levels - 1)            # bounded to (0, levels - 1)
    r = u + (u.round() - u).detach()                      # straight-through round to an integer level
    digits = r.detach().round().long()                    # 0 .. levels-1 per dim
    ids = (digits * levels ** torch.arange(z.shape[1])).sum(1)
    return r / (levels - 1) * 2 - 1, ids


def test_fsq(fn):
    z = torch.randn(1000, 4, requires_grad=True)
    z_q, ids = fn(z, levels=8)
    assert z_q.abs().max() <= 1.0 + 1e-6
    assert all(v.unique().numel() <= 8 for v in z_q.detach().T)       # at most 8 values per dim
    assert ids.min() >= 0 and ids.max() < 8**4
    z_q.sum().backward()
    assert z.grad.abs().sum() > 0                                     # gradient flows through the rounding


test_fsq(fsq)
z_q, ids = fsq(torch.randn(5, 4) * 2)
print("FSQ values per dim:", sorted(set(z_q.detach().flatten().tolist()))[:4], "... ids:", ids.tolist())

## Things to be able to explain

- Why can't gradients pass through `argmin`, and what exactly does straight-through pretend?
- VQGAN adds a perceptual loss and a GAN discriminator on top of VQ-VAE. Why does plain MSE give blurry images?
- Janus uses **two** vision encoders: SigLIP features for understanding and VQ tokens for generation. Why might one
  encoder be bad at both jobs? (Understanding wants semantic features; generation wants tokens that reconstruct
  pixels.)

## Drill

In [ ]:
def my_vq_forward(z_e, codebook, beta=0.25):
    raise NotImplementedError


drill(test_vq, my_vq_forward)